# VascuMap3D pipeline

Segments every 3D stack of the LIF files in one folder, then builds and measures the vessel graph of each mask. Outputs are written next to the input files.

## Input parameters

In [ ]:
# Path to the directory containing the LIF files
root_directory = r"/path/to/lif/folder"

# Index of the Brightfield channel, leave to None to automatically find it from lif metadata.
bf_index=None 

# Index of the fluo channel to segment from, leave to None if you want segmentation from Brightfield
fluo_index=None

# Trained models: Pix2Pix 3D (brightfield -> fluorescence) and the 2D U-Net
model_p2p_path = r"/path/to/translation.ckpt"
model_smp_path = r"/path/to/segmentation.pth"

# Crop profile to use, created with new_crop_profile.ipynb (ignored if crop=False)
# The same crop profile will be used for all the images from all the LIF files in the root directory.
# If you need different crop profiles for different LIF files, consider splitting in different subfolders
# and running the script on each subfolder separately.
crop_profile='my_profile' 

# Folder holding the crop profiles (<name>_rect.npy and <name>_fiducials.npy)
crop_profile_dir = r"/path/to/crop_profiles"

# Z range to crop the stack to, in micrometers. Choose a value that approximates the thickness 
# of the vascular network. Minimum is 160um. If the Z stack range is less than z_range, but superior 
# to 160um, the stack will be cropped to the Z stack range.
z_range=200

# Whether to perform 2.5D segmentation, recommended. 
# Set to False to iteratively perform 2D segmentation on each Z plane.
ortho=True 

# Save all stacks to the root directory (Brightfield, Fluo, Predicted, Mask, Probability). 
# Set to True to save all stacks, set to False to save only the final mask
save_all_stacks=False 

# Skeletonization method, 'thinning' or 'contraction'. The 'contraction' method is
# more sophisticated but slower and fails on thick networks.
skel_method='thinning' 

# Time limit in seconds for the contraction method, ignored if skel_method='thinning'
contraction_timelim=5*60 

# Save vessel and junction local metrics in .hdf5 files
save_local_metrics=True 

# Visualize the graph analysis
visualization=True 

# Whether to perform cropping of the images to a smaller region of interest.
# If False, the images will not be cropped.
crop = False

## Vessel segmentation

In [ ]:
import sys
sys.path.insert(0, "..")  # repository root, if the package is not installed
from vascumap.pipeline import run, process_directory

run(
    input_dir=root_directory,
    pattern="*.lif",
    ortho=ortho,
    bf_index=bf_index,
    fluo_index=fluo_index,
    crop_profile=crop_profile,
    crop_profile_dir=crop_profile_dir,
    z_range=z_range,
    save_all_stacks=save_all_stacks,
    crop=crop,
    model_p2p_path=model_p2p_path,
    model_smp_path=model_smp_path,
    )

## Graph analysis

In [ ]:
process_directory(
    masks_dir_path=root_directory,
    suffix="_vessel_mask",
    save_local_metrics=save_local_metrics,
    visualization=visualization,
    skel_method=skel_method,
    contraction_timelim=contraction_timelim,
    voxel_size_um=2.0,  # masks from run() are isotropic 2 um
    )